# AIO Dataset Preprocessing: CaucaFall + MCFD
## Features: YOLOv11-Pose + 9 PIFR Geometric Angles → (60, 60) Output

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os, gc, cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# ============================================================
# CONFIGURATION
# ============================================================

# Dataset paths
CAUCAFALL_DIR = "/kaggle/input/caucafall/Dataset CAUCAFall/CAUCAFall"
MCFD_DIR = "/kaggle/input/multiple-cameras-fall-dataset/dataset/dataset"
MCFD_CSV = "/kaggle/input/multiple-cameras-fall-dataset/data_tuple3.csv"

# Output directory
OUTPUT_DIR = "/kaggle/working/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# YOLO config
YOLO_MODEL = "yolo11n-pose.pt"
CONF_THRESHOLD = 0.5

# Temporal standardization
MAX_FRAMES = 120  # Truncate to first 120 frames
TARGET_FRAMES = 60  # Final output: 60 frames

# COCO keypoint indices (used for PIFR features)
COCO_IDX = {
    'nose': 0,
    'left_shoulder': 5,
    'right_shoulder': 6,
    'left_hip': 11,
    'right_hip': 12,
    'left_knee': 13,
    'right_knee': 14,
    'left_ankle': 15,
    'right_ankle': 16
}

## 1. PIFR Feature Extraction Functions

In [ ]:
# ============================================================
# FEATURE EXTRACTION: YOLOv11-Pose + 9 PIFR Angles
# ============================================================

def extract_keypoints(frame, model, frame_width, frame_height):
    """
    Run YOLOv11-Pose on a single frame and extract NORMALIZED keypoints.
    
    Returns:
        numpy array (17, 3) with [normalized_x, normalized_y, confidence]
        or None if no person detected.
    """
    results = model(frame, verbose=False, conf=CONF_THRESHOLD)
    
    if results[0].keypoints is None or len(results[0].keypoints) == 0:
        return None
    
    keypoints = results[0].keypoints.data[0].cpu().numpy()
    
    if len(keypoints) < 17:
        return None
    
    # Normalize: x/width, y/height → range [0, 1]
    normalized = np.zeros((17, 3), dtype=np.float32)
    for i in range(17):
        x, y, conf = keypoints[i]
        normalized[i, 0] = x / frame_width
        normalized[i, 1] = y / frame_height
        normalized[i, 2] = conf
    
    return normalized


def compute_9_pifr_features(keypoints):
    """
    Compute 9 geometric features from 17 NORMALIZED COCO keypoints.
    
    F1 & F2: Mean X/Y of all 17 keypoints (Center of Mass)
    F3: Shoulder-Nose Angle (BA vs BC)
    F4: Torso Angle (mid_hip → nose, y-component)
    F5: Hip Angle (l_hip → r_hip, x-component)
    F6: Shoulder Angle (l_shoulder → r_shoulder, x-component)
    F7: Left Leg Angle (hip→knee vs knee→ankle)
    F8: Right Leg Angle (same as F7, right side)
    F9: Nose-to-Ankle Angle (mid_ankle → nose, y-component)
    
    Returns: numpy array (9,) of geometric angles
    """
    features = []
    
    # F1 & F2: Center of Mass
    features.append(np.mean(keypoints[:, 0]))  # Mean X
    features.append(np.mean(keypoints[:, 1]))  # Mean Y
    
    # Extract key points
    nose = keypoints[COCO_IDX['nose']][:2]
    l_shoulder = keypoints[COCO_IDX['left_shoulder']][:2]
    r_shoulder = keypoints[COCO_IDX['right_shoulder']][:2]
    l_hip = keypoints[COCO_IDX['left_hip']][:2]
    r_hip = keypoints[COCO_IDX['right_hip']][:2]
    l_knee = keypoints[COCO_IDX['left_knee']][:2]
    r_knee = keypoints[COCO_IDX['right_knee']][:2]
    l_ankle = keypoints[COCO_IDX['left_ankle']][:2]
    r_ankle = keypoints[COCO_IDX['right_ankle']][:2]
    
    # F3: Shoulder-Nose Angle (BA = A-B, BC = C-B where B=Nose)
    BA = l_shoulder - nose
    BC = r_shoulder - nose
    norm_BA = np.linalg.norm(BA)
    norm_BC = np.linalg.norm(BC)
    if norm_BA > 0 and norm_BC > 0:
        dot_val = np.dot(BA, BC) / (norm_BA * norm_BC)
        F3 = np.arccos(np.clip(dot_val, -1.0, 1.0))
    else:
        F3 = 0.0
    features.append(F3)
    
    # F4: Torso Angle (mid_hip → nose)
    mid_hip = (l_hip + r_hip) / 2
    v_torso = mid_hip - nose
    norm_v_torso = np.linalg.norm(v_torso)
    if norm_v_torso > 0:
        F4 = np.arccos(np.clip(v_torso[1] / norm_v_torso, -1.0, 1.0))
    else:
        F4 = 0.0
    features.append(F4)
    
    # F5: Hip Angle (l_hip → r_hip, x-component)
    v_hip = r_hip - l_hip
    norm_v_hip = np.linalg.norm(v_hip)
    if norm_v_hip > 0:
        F5 = np.arccos(np.clip(v_hip[0] / norm_v_hip, -1.0, 1.0))
    else:
        F5 = 0.0
    features.append(F5)
    
    # F6: Shoulder Angle (l_shoulder → r_shoulder, x-component)
    v_shoulder = r_shoulder - l_shoulder
    norm_v_shoulder = np.linalg.norm(v_shoulder)
    if norm_v_shoulder > 0:
        F6 = np.arccos(np.clip(v_shoulder[0] / norm_v_shoulder, -1.0, 1.0))
    else:
        F6 = 0.0
    features.append(F6)
    
    # F7: Left Leg Angle (hip→knee vs knee→ankle)
    v1_left = l_knee - l_hip
    v2_left = l_ankle - l_knee
    norm_v1_left = np.linalg.norm(v1_left)
    norm_v2_left = np.linalg.norm(v2_left)
    if norm_v1_left > 0 and norm_v2_left > 0:
        dot_left = np.dot(v1_left, v2_left) / (norm_v1_left * norm_v2_left)
        F7 = np.arccos(np.clip(dot_left, -1.0, 1.0))
    else:
        F7 = 0.0
    features.append(F7)
    
    # F8: Right Leg Angle (hip→knee vs knee→ankle)
    v1_right = r_knee - r_hip
    v2_right = r_ankle - r_knee
    norm_v1_right = np.linalg.norm(v1_right)
    norm_v2_right = np.linalg.norm(v2_right)
    if norm_v1_right > 0 and norm_v2_right > 0:
        dot_right = np.dot(v1_right, v2_right) / (norm_v1_right * norm_v2_right)
        F8 = np.arccos(np.clip(dot_right, -1.0, 1.0))
    else:
        F8 = 0.0
    features.append(F8)
    
    # F9: Nose-to-Ankle Angle (mid_ankle → nose, y-component)
    mid_ankle = (l_ankle + r_ankle) / 2
    v_nose_to_ankle = mid_ankle - nose
    norm_v_na = np.linalg.norm(v_nose_to_ankle)
    if norm_v_na > 0:
        F9 = np.arccos(np.clip(v_nose_to_ankle[1] / norm_v_na, -1.0, 1.0))
    else:
        F9 = 0.0
    features.append(F9)
    
    return np.array(features, dtype=np.float32)


def extract_pifr_60d(keypoints):
    """
    Extract full 60D PIFR feature vector.
    
    51 keypoint values (17 × 3) + 9 geometric angles = 60D
    Returns: numpy array (60,)
    """
    # Flatten 17 keypoints (x, y, conf for each) → 51 values
    flattened_kpts = keypoints.flatten()  # shape: (51,)
    
    # Compute 9 geometric angles
    geometric = compute_9_pifr_features(keypoints)  # shape: (9,)
    
    # Concatenate → 60D vector
    return np.concatenate([flattened_kpts, geometric])

## 2. Temporal Standardization Functions

In [ ]:
# ============================================================
# TEMPORAL STANDARDIZATION: Ensure EXACT (60, 60) shape
# ============================================================

def standardize_to_60x60(video_features):
    """
    Standardize video feature sequence to EXACT shape (60, 60).
    
    Steps:
    1. Truncate: Only take first 120 frames
    2. Subsample: Take every 2nd frame (index 0, 2, 4, ...) → max 60 frames
    3. Pad: Repeat last frame if < 60 frames
    
    Returns: numpy array with EXACT shape (60, 60)
    """
    if video_features is None or len(video_features) == 0:
        return np.zeros((TARGET_FRAMES, 60), dtype=np.float32)
    
    video_features = np.array(video_features, dtype=np.float32)
    
    # Step 1: Truncate to first 120 frames
    if len(video_features) > MAX_FRAMES:
        video_features = video_features[:MAX_FRAMES]
    
    # Step 2: Subsample every 2nd frame → max 60 frames
    video_features = video_features[::2]
    
    # Step 3: Pad with last frame if < 60
    if len(video_features) < TARGET_FRAMES:
        last_frame = video_features[-1]
        padding_count = TARGET_FRAMES - len(video_features)
        padding = np.tile(last_frame, (padding_count, 1))
        video_features = np.vstack([video_features, padding])
    
    # Assert EXACT shape
    assert video_features.shape == (TARGET_FRAMES, 60), \
        f"Expected ({TARGET_FRAMES}, 60), got {video_features.shape}"
    
    return video_features

## 3. Video Processing Functions

In [ ]:
# ============================================================
# VIDEO PROCESSING: Full video and Segmented video
# ============================================================

def process_video_full(video_path, model, fallback_vector):
    """
    Process ENTIRE video and extract PIFR features.
    
    Args:
        video_path: Path to .avi file
        model: YOLO pose model
        fallback_vector: 60D vector for missing detections
    Returns:
        List of (60,) vectors, one per frame
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return None
    
    frame_features = []
    prev_vector = fallback_vector
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_height, frame_width = frame.shape[:2]
        keypoints = extract_keypoints(frame, model, frame_width, frame_height)
        
        if keypoints is None:
            # No detection: duplicate previous vector
            if prev_vector is not None:
                frame_features.append(prev_vector.copy())
            else:
                frame_features.append(np.zeros(60, dtype=np.float32))
        else:
            pifr_vec = extract_pifr_60d(keypoints)
            frame_features.append(pifr_vec)
            prev_vector = pifr_vec
    
    cap.release()
    
    if len(frame_features) == 0:
        return None
    
    return frame_features


def process_video_segment(video_path, start_frame, end_frame, model, fallback_vector):
    """
    Process SPECIFIC SEGMENT of video (for MCFD).
    
    Args:
        video_path: Path to .avi file
        start_frame: Starting frame index (inclusive)
        end_frame: Ending frame index (inclusive)
        model: YOLO pose model
        fallback_vector: 60D vector for missing detections
    Returns:
        List of (60,) vectors for the segment
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return None
    
    # Skip to start frame using cv2 CAP_PROP_POS_FRAMES
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    frame_features = []
    prev_vector = fallback_vector
    current_frame = start_frame
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if current_frame > end_frame:
            break
        
        frame_height, frame_width = frame.shape[:2]
        keypoints = extract_keypoints(frame, model, frame_width, frame_height)
        
        if keypoints is None:
            if prev_vector is not None:
                frame_features.append(prev_vector.copy())
            else:
                frame_features.append(np.zeros(60, dtype=np.float32))
        else:
            pifr_vec = extract_pifr_60d(keypoints)
            frame_features.append(pifr_vec)
            prev_vector = pifr_vec
        
        current_frame += 1
    
    cap.release()
    
    if len(frame_features) == 0:
        return None
    
    return frame_features

## 4. Dataset 1: CaucaFall Processing

In [ ]:
# ============================================================
# DATASET 1: CAUCAFALL
# ============================================================

def is_fall_action(folder_name):
    """Label = 1 if 'fall' in folder name (case-insensitive)."""
    return 1 if "fall" in folder_name.lower() else 0


def safe_filename(name):
    """Sanitize filename by replacing special characters."""
    return name.replace(".", "_").replace(" ", "_").replace("/", "_").replace("\\", "_")


def process_caucafall(model):
    """
    Process CaucaFall dataset.
    
    Structure: Subject.*/ActionFolder/*.avi
    Label: 1 if 'fall' in ActionFolder, else 0
    Output: /kaggle/working/processed/X_cauca_{Subject}_{Action}.npy
    """
    print("\n" + "="*60)
    print("Processing: CaucaFall Dataset")
    print("="*60)
    
    zero_fallback = np.zeros(60, dtype=np.float32)
    
    # Collect all videos
    video_list = []
    for subject_dir in sorted(os.listdir(CAUCAFALL_DIR)):
        subject_path = os.path.join(CAUCAFALL_DIR, subject_dir)
        
        if not os.path.isdir(subject_path) or not subject_dir.startswith("Subject."):
            continue
        
        for action_folder in sorted(os.listdir(subject_path)):
            action_path = os.path.join(subject_path, action_folder)
            
            if not os.path.isdir(action_path):
                continue
            
            # Find all .avi files
            avi_files = [f for f in os.listdir(action_path) if f.endswith('.avi')]
            if len(avi_files) == 0:
                continue
            
            for avi_file in avi_files:
                video_path = os.path.join(action_path, avi_file)
                label = is_fall_action(action_folder)
                
                video_list.append({
                    'subject': subject_dir,
                    'action': action_folder,
                    'video_path': video_path,
                    'label': label
                })
    
    print(f"Found {len(video_list)} videos to process")
    
    # Process each video
    processed = skipped = errors = 0
    
    for item in tqdm(video_list, desc="CaucaFall"):
        # Generate output filenames
        safe_subj = safe_filename(item['subject'])
        safe_action = safe_filename(item['action'])
        
        x_filename = f"X_cauca_{safe_subj}_{safe_action}.npy"
        y_filename = f"y_cauca_{safe_subj}_{safe_action}.npy"
        
        x_path = os.path.join(OUTPUT_DIR, x_filename)
        y_path = os.path.join(OUTPUT_DIR, y_filename)
        
        # Skip if already processed
        if os.path.exists(x_path) and os.path.exists(y_path):
            skipped += 1
            continue
        
        try:
            # Process entire video
            features = process_video_full(item['video_path'], model, zero_fallback)
            
            if features is None or len(features) == 0:
                print(f"Warning: Empty video {item['video_path']}")
                errors += 1
                continue
            
            # Standardize to exact (60, 60)
            features = standardize_to_60x60(features)
            
            # Save
            np.save(x_path, features)
            np.save(y_path, np.array([item['label']], dtype=np.int32))
            
            processed += 1
            
        except Exception as e:
            print(f"Error: {item['subject']}/{item['action']}: {e}")
            errors += 1
        
        finally:
            # Memory cleanup
            if 'features' in locals():
                del features
            gc.collect()
    
    print(f"\nCaucaFall Summary: Processed={processed}, Skipped={skipped}, Errors={errors}")
    return processed, skipped, errors

## 5. Dataset 2: MCFD Processing

In [ ]:
# ============================================================
# DATASET 2: MCFD (Multiple Cameras Fall Dataset)
# ============================================================

def process_mcfd(model):
    """
    Process MCFD dataset based on CSV annotations.
    
    CSV columns: chute, cam, start, end, label (delimiter=',')
    Video path: chute{chute:02d}/cam{cam}.avi
    Slicing: Only process frames from start to end
    Output: /kaggle/working/processed/X_mcfd_c{chute}_cam{cam}_row{index}.npy
    """
    print("\n" + "="*60)
    print("Processing: MCFD Dataset")
    print("="*60)
    
    zero_fallback = np.zeros(60, dtype=np.float32)
    
    # Load CSV annotations
    df = pd.read_csv(MCFD_CSV)
    print(f"Found {len(df)} annotations in CSV")
    
    # Process each segment
    processed = skipped = errors = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="MCFD"):
        chute = int(row['chute'])
        cam = int(row['cam'])
        start = int(row['start'])
        end = int(row['end'])
        label = int(row['label'])
        
        # Build video path
        chute_folder = f"chute{chute:02d}"
        video_filename = f"cam{cam}.avi"
        video_path = os.path.join(MCFD_DIR, chute_folder, video_filename)
        
        # Check if video exists
        if not os.path.exists(video_path):
            errors += 1
            continue
        
        # Generate output filenames
        x_filename = f"X_mcfd_c{chute:02d}_cam{cam}_row{idx}.npy"
        y_filename = f"y_mcfd_c{chute:02d}_cam{cam}_row{idx}.npy"
        
        x_path = os.path.join(OUTPUT_DIR, x_filename)
        y_path = os.path.join(OUTPUT_DIR, y_filename)
        
        # Skip if already processed
        if os.path.exists(x_path) and os.path.exists(y_path):
            skipped += 1
            continue
        
        try:
            # Process specific segment [start, end]
            features = process_video_segment(video_path, start, end, model, zero_fallback)
            
            if features is None or len(features) == 0:
                print(f"Warning: Empty segment chute{chute}/cam{cam} row{idx}")
                errors += 1
                continue
            
            # Standardize to exact (60, 60)
            features = standardize_to_60x60(features)
            
            # Save
            np.save(x_path, features)
            np.save(y_path, np.array([label], dtype=np.int32))
            
            processed += 1
            
        except Exception as e:
            print(f"Error: chute{chute}/cam{cam} row{idx}: {e}")
            errors += 1
        
        finally:
            # Memory cleanup
            if 'features' in locals():
                del features
            gc.collect()
    
    print(f"\nMCFD Summary: Processed={processed}, Skipped={skipped}, Errors={errors}")
    return processed, skipped, errors

## 6. Main Execution

In [ ]:
# ============================================================
# MAIN: PROCESS BOTH DATASETS
# ============================================================

def main():
    """Process both datasets and create AIO dataset."""
    
    print("="*60)
    print("AIO FALL DETECTION DATASET PREPROCESSING")
    print("Datasets: CaucaFall + MCFD")
    print("Output Shape: (60, 60) for ALL files")
    print("="*60)
    
    # Load YOLO model once
    print("\nLoading YOLOv11-Pose model...")
    model = YOLO(YOLO_MODEL)
    print("Model loaded successfully!")
    
    # Process Dataset 1: CaucaFall
    cauca_processed, cauca_skipped, cauca_errors = process_caucafall(model)
    
    # Process Dataset 2: MCFD
    mcfd_processed, mcfd_skipped, mcfd_errors = process_mcfd(model)
    
    # Final Summary
    total_processed = cauca_processed + mcfd_processed
    total_skipped = cauca_skipped + mcfd_skipped
    total_errors = cauca_errors + mcfd_errors
    
    print("\n" + "="*60)
    print("AIO PREPROCESSING COMPLETE")
    print("="*60)
    print(f"  Total Processed: {total_processed}")
    print(f"  Total Skipped:  {total_skipped}")
    print(f"  Total Errors:   {total_errors}")
    print(f"  Output Dir:     {OUTPUT_DIR}")
    print(f"  File Shape:     (60, 60)")
    print("="*60)
    
    # Count output files
    x_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('X_')]
    y_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('y_')]
    print(f"\nOutput files: {len(x_files)} X files, {len(y_files)} y files")


# Run
if __name__ == "__main__":
    main()